# Spiking Neural Network Transformer for Time Series Forecasting
## Dataset: S&P 500

**Architecture:** Spike-Driven Transformer (Yao et al., 2024)
adapted for univariate time series forecasting.

**Reference:** *Spike-driven Transformer V2: Meta Spiking Neural Network Architecture
Inspiring the Design of Next-generation Neuromorphic Chips* — ICLR 2024

This notebook is a **controlled architectural substitution** of the LSTM-SNP model.
All preprocessing, training protocol, and evaluation methodology are identical
to the LSTM-SNP experiments — only the model architecture changes.

### Key Equations (from the paper)

**LIF Neuron (Eq 1-3):**
- $U[t] = H[t-1] + X[t]$  — membrane potential accumulation
- $S[t] = \text{Hea}(U[t] - u_{th})$  — spike generation (Heaviside)
- $H[t] = V_{reset} \cdot S[t] + (\beta \cdot U[t]) \cdot (1 - S[t])$  — temporal output with reset

**SDSA (Eq 14):**
- $\text{SDSA}(Q, K, V) = \mathcal{SN}(\text{SUM}_c(Q_S \otimes K_S)) \otimes V_S$

**Membrane Shortcut (Eq 9, 11):**
- $U'_l = \text{SDSA}(S_{l-1}) + U_{l-1}$
- $S_l = \mathcal{SN}(\text{MLP}(S'_l) + U'_l)$


In [ ]:
# ============================================================
# GLOBAL IMPORTS (Consolidated)
# ============================================================
from IPython.display import display
from math import sqrt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import platform
import time
import time as _timer_module
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings

# ============================================================
# GLOBAL CONFIGURATION (CUDA & NOISE)
# ============================================================
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0' # Enable RTX 6000 Pro
import torch
if torch.cuda.is_available():
    torch.cuda.set_device(0)

noise_levels = [0.0, 0.005, 0.05, 0.10]
_molab_results = []

def add_gaussian_noise(series, noise_level):
    if noise_level == 0.0:
        return series
    noise = np.random.normal(0, noise_level, len(series))
    return series + noise
# ============================================================

# ============================================================
# FINAL RESULTS TABLE ACROSS ALL NOISE LEVELS
# ============================================================
import pandas as pd
from tabulate import tabulate
if len(_molab_results) > 0:
    df_res = pd.DataFrame(_molab_results, columns=['Noise Level (lambda)', 'RMSE', 'MSE', 'NMSE'])
    print('\n' + '='*80)
    print('FINAL RESULTS TABLE ACROSS ALL 4 NOISE LEVELS')
    print('='*80)
    print(tabulate(df_res, headers='keys', tablefmt='github', showindex=False))


In [ ]:
# ============================================================
# GLOBAL IMPORTS (Consolidated)
# ============================================================

# ============================================================
# GLOBAL CONFIGURATION (CUDA & NOISE)
# ============================================================
os.environ['CUDA_VISIBLE_DEVICES'] = '0' # Enable RTX 6000 Pro
if torch.cuda.is_available():
    torch.cuda.set_device(0)

noise_levels = [0.0, 0.005, 0.05, 0.10]
_molab_results = []

def add_gaussian_noise(series, noise_level):
    if noise_level == 0.0:
        return series
    noise = np.random.normal(0, noise_level, len(series))
    return series + noise
# ============================================================

# ============================================================
# FINAL RESULTS TABLE ACROSS ALL NOISE LEVELS
# ============================================================
import pandas as pd
from tabulate import tabulate
if len(_molab_results) > 0:
    df_res = pd.DataFrame(_molab_results, columns=['Noise Level (lambda)', 'RMSE', 'MSE', 'NMSE'])
    print('\n' + '='*80)
    print('FINAL RESULTS TABLE ACROSS ALL 4 NOISE LEVELS')
    print('='*80)
    print(tabulate(df_res, headers='keys', tablefmt='github', showindex=False))


In [ ]:
# ============================================================
# PROCESS IDENTIFICATION
# ============================================================
print(f"Process ID (PID): {os.getpid()}")


In [ ]:
# ============================================================
# NOTEBOOK TIMER — START
# ============================================================
_NOTEBOOK_START_TIME = _timer_module.time()
print(f"Notebook execution started at: {_timer_module.strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# Spiking Neural Network Transformer for Time Series Forecasting
## Dataset: Lake Erie
## + Gaussian Input-Noise Robustness (0.5% / 5% / 10% / 15%)

**Architecture:** Spike-Driven Transformer (Yao et al., 2024)
adapted for univariate time series forecasting.

**Reference:** *Spike-driven Transformer V2: Meta Spiking Neural Network Architecture
Inspiring the Design of Next-generation Neuromorphic Chips* — ICLR 2024

This notebook is a **controlled architectural substitution** of the LSTM-SNP model.
All preprocessing, training protocol, and evaluation methodology are identical
to the LSTM-SNP experiments — only the model architecture changes.

This version additionally injects Gaussian noise into the input features at test time:

x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = eta * std(x)

Noise is injected into input features only (never targets), so the resulting metrics reflect
model sensitivity to input perturbation, not label corruption. This mirrors the protocol used
in the LSTM-SNP baseline and FuzzyGateReplacement noise-robustness notebooks, so results are
directly comparable across all variants.

This version also adds: validation-based early stopping, gradient clipping (always applied,
for training stability parity with the other variants), an NMSE denominator computed against
the actual values (the original cell used `predictions - meanV`, which is corrected here to
`actual - meanV` for cross-variant comparability), and noise-sweep averaging over multiple
ε(t) draws with a paired per-model degradation test.

### Key Equations (from the paper)

**LIF Neuron (Eq 1-3):**
- $U[t] = H[t-1] + X[t]$  — membrane potential accumulation
- $S[t] = \text{Hea}(U[t] - u_{th})$  — spike generation (Heaviside)
- $H[t] = V_{reset} \cdot S[t] + (\beta \cdot U[t]) \cdot (1 - S[t])$  — temporal output with reset

**SDSA (Eq 14):**
- $\text{SDSA}(Q, K, V) = \mathcal{SN}(\text{SUM}_c(Q_S \otimes K_S)) \otimes V_S$

**Membrane Shortcut (Eq 9, 11):**
- $U'_l = \text{SDSA}(S_{l-1}) + U_{l-1}$
- $S_l = \mathcal{SN}(\text{MLP}(S'_l) + U'_l)$

In [ ]:
# ============================================================
# ALL IMPORTS
# ============================================================
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if hasattr(torch.backends, 'mps'):
    print(f"MPS available: {torch.backends.mps.is_available()}")

In [ ]:
# ============================================================
# PREPROCESSING — Identical to LSTM-SNP Pipeline
# ============================================================
# 1. First-order differencing
# 2. Lag-1 supervised learning format
# 3. Train-test split (last 60 = test)
# 4. MinMaxScaler [-1, 1] (fit on train only)
# 5. Reshape to (samples, 1, 1)

def difference(dataset, interval=1):
    """First-order differencing: diff(t) = raw(t) - raw(t-interval)"""
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)


def timeseries_to_supervised(data, lag=1):
    """Convert to supervised format: X(t)=data(t-lag), y(t)=data(t). NaN filled with 0."""
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag + 1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values


def prepare_data(raw_values, n_test=60):
    """Full preprocessing pipeline (identical to LSTM-SNP notebooks)."""
    # Step 1: First-order differencing
    diff_values = difference(raw_values, 1)

    # Step 2: Convert to supervised format (lag=1)
    supervised = timeseries_to_supervised(diff_values, 1)

    # Step 3: Train-test split
    train, test = supervised[:-n_test], supervised[-n_test:]

    # Step 4: Scale to [-1, 1] — fit on train only
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(train)
    train_scaled = scaler.transform(train)
    test_scaled = scaler.transform(test)

    # Step 5: Split into X, y
    X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
    X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]

    # Step 6: Reshape X for model input: (samples, 1, features=1)
    X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
    X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

    return X_train, y_train, X_test, y_test, scaler, raw_values

In [ ]:
# ============================================================
# SPIKING NEURON — Leaky Integrate-and-Fire (LIF)
# ============================================================
# Faithfully implements Equations 1-3 from the Spike-Driven Transformer paper:
#
#   Eq 1: U[t] = H[t-1] + X[t]                              (membrane potential)
#   Eq 2: S[t] = Hea(U[t] - u_th)                           (spike generation)
#   Eq 3: H[t] = V_reset * S[t] + (β * U[t]) * (1 - S[t])  (temporal output)
#
# Where:
#   U[t] = membrane potential at timestep t
#   H[t] = temporal output (decayed potential or reset)
#   S[t] = binary spike output {0, 1}
#   β < 1 = decay factor
#   u_th = firing threshold
#   V_reset = reset potential after spike
#   Hea(·) = Heaviside step function
#
# Surrogate gradient (for backpropagation):
#   ∂S/∂U ≈ 1 / (1 + α|U - u_th|)²


class SurrogateHeaviside(torch.autograd.Function):
    """Heaviside step with surrogate gradient for backpropagation."""
    @staticmethod
    def forward(ctx, input, alpha):
        ctx.save_for_backward(input)
        ctx.alpha = alpha
        return (input >= 0).float()

    @staticmethod
    def backward(ctx, grad_output):
        (input,) = ctx.saved_tensors
        alpha = ctx.alpha
        grad_input = grad_output / (1 + alpha * input.abs()) ** 2
        return grad_input, None


def surrogate_heaviside(x, alpha=2.0):
    return SurrogateHeaviside.apply(x, alpha)


class LIFNeuron(nn.Module):
    """
    Leaky Integrate-and-Fire neuron (Eq 1-3 from the paper).

    Processes input across T spiking timesteps.
    Membrane potential H is maintained across timesteps, reset per forward call.
    """
    def __init__(self, beta=0.5, v_th=0.5, v_reset=0.0, alpha=2.0):
        super().__init__()
        self.beta = beta        # decay factor (β < 1)
        self.v_th = v_th        # firing threshold (u_th)
        self.v_reset = v_reset  # reset potential (V_reset)
        self.alpha = alpha      # surrogate gradient sharpness

    def forward(self, x_seq):
        """
        x_seq: (batch, T, features) — input current across T timesteps.
        Returns: (batch, T, features) — binary spike output.
        """
        batch, T, features = x_seq.shape
        device = x_seq.device

        H = torch.zeros(batch, features, device=device)  # temporal output H[t-1]
        spikes = []

        for t in range(T):
            # Eq 1: U[t] = H[t-1] + X[t]
            U = H + x_seq[:, t, :]

            # Eq 2: S[t] = Hea(U[t] - u_th)
            S = surrogate_heaviside(U - self.v_th, self.alpha)

            # Eq 3: H[t] = V_reset * S[t] + (β * U[t]) * (1 - S[t])
            H = self.v_reset * S + (self.beta * U) * (1 - S.detach())

            spikes.append(S)

        return torch.stack(spikes, dim=1)  # (batch, T, features)

In [ ]:
# ============================================================
# SPIKE-DRIVEN SELF-ATTENTION (SDSA)
# ============================================================
# Implements Equations 14-16 from the Spike-Driven Transformer paper.
#
# Standard Self-Attention (Eq 13):
#   VSA(Q,K,V) = softmax(QK^T / √d) · V     → O(N²D + N²D)
#
# Spike-Driven Self-Attention (Eq 14):
#   SDSA(Q,K,V) = SN(SUM_c(Q_S ⊗ K_S)) ⊗ V_S
#
# Where:
#   Q, K, V = linear projections of spike input S (float-point)
#   Q_S, K_S, V_S = SN(Q), SN(K), SN(V)  (converted to spikes)
#   ⊗ = Hadamard product (element-wise multiplication)
#   SUM_c = column sum (sum over token/temporal dimension)
#   SN = Spiking Neuron (LIF)
#
# The key insight: since spikes are binary {0, 1}, the Hadamard product
# Q_S ⊗ K_S is equivalent to a mask operation. SUM_c aggregates activity
# across tokens to produce a D-dimensional feature attention mask.
# This mask gates V_S, selecting relevant feature channels.
#
# Computational complexity: O(ND) — linear in both N and D.


class SpikeDrivenSelfAttention(nn.Module):
    """
    SDSA adapted for 1D time series.

    Input: spike tensor S ∈ {0,1}^(batch, T, D)
    Output: spike-gated features ∈ R^(batch, T, D)
    """
    def __init__(self, d_model, n_heads, beta=0.5, v_th=0.5):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # Linear projections (float-point, Eq 14 setup)
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.proj = nn.Linear(d_model, d_model)

        # Spiking neurons to convert projections to spike tensors
        self.sn_q = LIFNeuron(beta=beta, v_th=v_th)
        self.sn_k = LIFNeuron(beta=beta, v_th=v_th)
        self.sn_v = LIFNeuron(beta=beta, v_th=v_th)
        # SN for attention mask (Eq 14: SN after SUM_c)
        self.sn_attn = LIFNeuron(beta=beta, v_th=v_th)

    def forward(self, spike_input):
        """
        spike_input: (batch, T, d_model) — spike tensor from previous layer.
        """
        batch, T, D = spike_input.shape

        # Linear projections (float-point)
        Q = self.W_Q(spike_input)
        K = self.W_K(spike_input)
        V = self.W_V(spike_input)

        # Convert to spike tensors via SN (spiking neuron)
        Q_S = self.sn_q(Q)  # (batch, T, D)
        K_S = self.sn_k(K)
        V_S = self.sn_v(V)

        # ── SDSA Computation (Eq 14) ──
        # Step 1: Hadamard product Q_S ⊗ K_S
        QK = Q_S * K_S  # (batch, T, D) — element-wise, spike masking

        # Step 2: SUM_c — sum over temporal dimension (columns)
        # Produces D-dimensional attention vector per batch
        attn_sum = QK.sum(dim=1, keepdim=True)  # (batch, 1, D)

        # Step 3: SN — convert attention map to binary mask
        attn_mask = self.sn_attn(attn_sum)  # (batch, 1, D)

        # Step 4: Hadamard with V_S — gate value spikes by attention mask
        output = attn_mask * V_S  # (batch, T, D) — broadcast across T

        # Output projection
        return self.proj(output)

In [ ]:
# ============================================================
# SPIKING TRANSFORMER BLOCK with Membrane Shortcuts
# ============================================================
# Implements Equations 8-12 from the paper.
#
# The block consists of SDSA + MLP with Membrane Shortcuts (MS):
#
#   Eq 8:  S_0 = SN(U_0)                          (initial spikes)
#   Eq 9:  U'_l = SDSA(S_{l-1}) + U_{l-1}         (MS on SDSA)
#   Eq 10: S'_l = SN(U'_l)
#   Eq 11: S_l = SN(MLP(S'_l) + U'_l)             (MS on MLP)
#   Eq 12: Y = CH(GAP(S_L))                        (output)
#
# Membrane Shortcut (MS) connects membrane potentials between layers,
# ensuring binary spikes are maintained throughout (unlike SEW shortcut).


class SpikingMLP(nn.Module):
    """Simple MLP for the spiking transformer block."""
    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.fc1 = nn.Linear(d_model, ff_dim)
        self.fc2 = nn.Linear(ff_dim, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class SpikingTransformerBlock(nn.Module):
    """
    Single Spiking Transformer block with Membrane Shortcuts (Eq 9-11).

    Takes both spike tensor S and membrane potential U from previous layer.
    Returns updated spike tensor and membrane potential.
    """
    def __init__(self, d_model, n_heads, ff_dim, beta=0.5, v_th=0.5):
        super().__init__()
        self.sdsa = SpikeDrivenSelfAttention(d_model, n_heads, beta, v_th)
        self.mlp = SpikingMLP(d_model, ff_dim)

        # SN after SDSA + membrane shortcut (Eq 10)
        self.sn_post_sdsa = LIFNeuron(beta=beta, v_th=v_th)
        # SN after MLP + membrane shortcut (Eq 11)
        self.sn_post_mlp = LIFNeuron(beta=beta, v_th=v_th)

    def forward(self, S_prev, U_prev):
        """
        S_prev: (batch, T, D) — spike tensor from previous layer
        U_prev: (batch, T, D) — membrane potential from previous layer
        """
        # Eq 9: U'_l = SDSA(S_{l-1}) + U_{l-1}  (Membrane Shortcut)
        U_prime = self.sdsa(S_prev) + U_prev

        # Eq 10: S'_l = SN(U'_l)
        S_prime = self.sn_post_sdsa(U_prime)

        # Eq 11: S_l = SN(MLP(S'_l) + U'_l)  (Membrane Shortcut on MLP)
        S_l = self.sn_post_mlp(self.mlp(S_prime) + U_prime)

        return S_l, U_prime

In [ ]:
# ============================================================
# SPIKING TRANSFORMER FORECASTER — Complete Model
# ============================================================
# Adapted from the Spike-Driven Transformer for univariate time series.
#
# Architecture:
#   1. Input Projection: scalar → d_model (replaces SPS from the paper)
#   2. Temporal Expansion: repeat across T spiking timesteps
#   3. Initial SN: S_0 = SN(U_0) (Eq 8)
#   4. L × SpikingTransformerBlock with Membrane Shortcuts (Eq 9-11)
#   5. GAP: Global Average Pooling over T timesteps (Eq 12)
#   6. Regression Head: d_model → 1
#
# Hyperparameters matched to LSTM-SNP:
#   d_model = 8 (same as LSTM-SNP hidden units)
#   n_heads = 2, n_layers = 1, ff_dim = 16, T = 4


class SpikingTransformerForecaster(nn.Module):
    def __init__(self, input_dim=1, d_model=8, n_heads=2, n_layers=1,
                 ff_dim=16, T=4, beta=0.5, v_th=0.5):
        super().__init__()
        self.T = T
        self.d_model = d_model

        # Input projection (replaces Patch Splitting Module from paper)
        self.input_proj = nn.Linear(input_dim, d_model)

        # Initial SN (Eq 8: S_0 = SN(U_0))
        self.sn_init = LIFNeuron(beta=beta, v_th=v_th)

        # Spiking Transformer blocks
        self.blocks = nn.ModuleList([
            SpikingTransformerBlock(d_model, n_heads, ff_dim, beta, v_th)
            for _ in range(n_layers)
        ])

        # Regression head (replaces Classification Head from paper)
        self.head = nn.Linear(d_model, 1)


    def reset_states(self, batch_size, device):
        self.u = torch.zeros(1, device=device)
        
    def detach_states(self):
        if hasattr(self, 'u') and self.u is not None:
            self.u = self.u.detach()

    def forward(self, x):
        """
        x: (batch, 1, 1) — single scaled differenced value
        Returns: (batch, 1) — predicted value
        """
        batch = x.shape[0]
        x = x.view(batch, -1)  # (batch, 1)

        # Input projection: (batch, 1) → (batch, d_model)
        u = self.input_proj(x)

        # Temporal expansion: repeat across T spiking timesteps
        # (analogous to repeating images T times in the paper)
        U_0 = u.unsqueeze(1).repeat(1, self.T, 1)  # (batch, T, d_model)

        # Eq 8: S_0 = SN(U_0)
        S = self.sn_init(U_0)
        U = U_0

        # Pass through L spiking transformer blocks (Eq 9-11)
        for block in self.blocks:
            S, U = block(S, U)

        # Eq 12: Y = CH(GAP(S_L))
        # GAP: Global Average Pooling over T timesteps
        gap = S.mean(dim=1)  # (batch, d_model)

        # Regression head
        output = self.head(gap)  # (batch, 1)
        return output


# Print model summary
model_test = SpikingTransformerForecaster()
total_params = sum(p.numel() for p in model_test.parameters())
print(f"SNN-Transformer parameters: {total_params}")
del model_test

In [ ]:
# ============================================================
# Model Wrapper (PyTorch)
# ============================================================
def build_model(input_dim=1, units=8):
    return SpikingTransformerForecaster(
        input_dim=input_dim, 
        d_model=units, 
        n_heads=2, 
        n_layers=1, 
        ff_dim=16, 
        T=4
    )

# Quick architecture check
model = build_model(input_dim=1, units=8)
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters())}")


In [1]:
# ============================================================
# 1. Load Time Series Data
# ============================================================
series = pd.read_csv(
    r'C:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\dataset\sp500.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)
raw_values = series.values.flatten()
print(f"Data shape: {raw_values.shape}")
print(f"First 5 values: {raw_values[:5]}")

NameError: name 'pd' is not defined

In [7]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

diff_values = difference(raw_values, 1)

In [8]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

supervised = timeseries_to_supervised(diff_values, 1)
print(f"Supervised data shape: {supervised.shape}")

Supervised data shape: (599, 2)


In [ ]:
# ============================================================
# 4. Train-Validation-Test Split (Last 60 points as Test)
# ============================================================
n = len(supervised)
test_size = 60
train_val = supervised[:n - test_size]
test = supervised[n - test_size:]

val_size = int(len(train_val) * 0.1)
train = train_val[:-val_size]
val = train_val[-val_size:]

print(f"Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")

# ============================================================
# 5. Feature Scaling
# ============================================================
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train)  # fit only on train to avoid leakage
train_scaled = scaler.transform(train)
val_scaled = scaler.transform(val)
test_scaled = scaler.transform(test)

In [10]:
# ============================================================
# 6. Reshape for RNN Input
# ============================================================

X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))

X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (479, 1, 1), y_train shape: (479,)


In [ ]:
# Gaussian noise injection (same protocol as the LSTM-SNP baseline / FuzzyGateReplacement notebooks)
ref_std = X_train.std()
print(f"Reference std(x) used for noise scaling (train inputs): {ref_std:.6f}")

NOISE_LEVELS = [0.005, 0.05, 0.10, 0.15]  # 0.5%, 5%, 10%, 15%

def add_gaussian_noise(X, noise_level, ref_std, seed=None):
    """
    x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = noise_level * std(x)
    Applied to input features X only -- never to targets/labels.
    """
    rng = np.random.default_rng(seed)
    sigma_eps = noise_level * ref_std
    eps = rng.normal(loc=0.0, scale=sigma_eps, size=X.shape)
    return X + eps

In [ ]:
# Evaluation function -- warms up the model's spiking/membrane state on clean training
# data, then runs single-step-ahead prediction on X_eval (clean or Gaussian-noise-corrupted
# input features). Test targets are always the clean, ground-truth values.
# NMSE denominator uses the actual series (corrected from the original `predictions - meanV`).

def evaluate_on_test(model, X_eval, train, train_scaled, raw_values, scaler, device):
    model.eval()
    model.reset_states(1, device)

    with torch.no_grad():
        for i in range(len(train_scaled)):
            X_raw = train_scaled[i, 0:-1]
            X_input = torch.tensor(X_raw, dtype=torch.float32).view(1, 1, len(X_raw)).to(device)
            model(X_input)

        predictions = []
        for i in range(len(X_eval)):
            X = X_eval[i]
            X_input = torch.tensor(X, dtype=torch.float32).view(1, 1, len(X)).to(device)
            yhat = model(X_input).item()

            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)

    actual = raw_values[-len(X_eval):]
    mse = mean_squared_error(actual, predictions)
    rmse = sqrt(mse)
    meanV = np.mean(actual)
    dominator = np.linalg.norm(np.array(actual) - meanV, 2)
    nmse = mse / np.power(dominator, 2)
    return predictions, rmse, mse, nmse

In [ ]:
# Training loop (60 runs, early stopping + gradient clipping + per-epoch save)

import os
import csv
import pickle
import time

CHECKPOINT_DIR = 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

EPOCH_LOG_PATH = os.path.join(CHECKPOINT_DIR, 'epoch_log.csv')
PROGRESS_PATH = os.path.join(CHECKPOINT_DIR, 'progress.pkl')

# Write CSV header once (if the log doesn't already exist)
if not os.path.exists(EPOCH_LOG_PATH):
    with open(EPOCH_LOG_PATH, 'w', newline='') as f:
        csv.writer(f).writerow(['run', 'epoch', 'train_loss', 'timestamp'])

def log_epoch(run, epoch, train_loss):
    """Append one epoch's result to disk immediately (flushed + fsynced)."""
    with open(EPOCH_LOG_PATH, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([run, epoch, train_loss, time.strftime('%Y-%m-%d %H:%M:%S')])
        f.flush()
        os.fsync(f.fileno())

def save_progress(run, epoch, run_losses_partial, max_retries=5, retry_delay=0.5):
    """
    Overwrite a resumable snapshot after every epoch: all fully-completed runs'
    results, plus the in-progress run's losses so far. If the kernel dies mid-run,
    you lose at most the current run's partial epochs, not everything.

    Retries on transient Windows file-lock errors (WinError 5), which happen
    when OneDrive/antivirus/indexing briefly holds the destination handle
    during os.replace. Never lets a checkpoint hiccup crash the training run.
    """
    snapshot = {
        'all_rmse': all_rmse,
        'all_mse': all_mse,
        'all_nmse': all_nmse,
        'all_predictions': all_predictions,
        'all_losses': all_losses,
        'current_run': run,
        'current_epoch': epoch,
        'current_run_losses_partial': run_losses_partial,
    }
    tmp_path = PROGRESS_PATH + '.tmp'
    with open(tmp_path, 'wb') as f:
        pickle.dump(snapshot, f)

    for attempt in range(max_retries):
        try:
            os.replace(tmp_path, PROGRESS_PATH)  # atomic on POSIX and Windows
            return
        except PermissionError:
            if attempt == max_retries - 1:
                print(f"[warn] save_progress: could not replace {PROGRESS_PATH} "
                      f"after {max_retries} attempts, skipping this checkpoint")
                try:
                    os.remove(tmp_path)
                except OSError:
                    pass
                return
            time.sleep(retry_delay)

all_rmse = []
all_mse = []
all_nmse = []
all_predictions = []
all_losses = []
all_models = []  # keep every trained model for later noise-robustness eval

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")

N_RUNS = 60
PATIENCE = 10
MIN_DELTA = 1e-5  # minimum improvement in train_loss to reset patience

for run in range(N_RUNS):
    print(f'\n===== RUN {run + 1}/{N_RUNS} =====')

    np.random.seed(run)
    torch.manual_seed(run)
    model = build_model(input_dim=1, units=8).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    run_losses = []

    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    n_samples = X_train_t.size(0)

    best_train_loss = float('inf')
    best_state = None
    patience_ctr = 0

    for epoch in range(100):
        model.train()
        model.reset_states(1, device)
        epoch_loss = 0.0

        for i in range(n_samples):
            x_i = X_train_t[i:i + 1]
            y_i = y_train_t[i:i + 1]

            optimizer.zero_grad()
            pred = model(x_i)
            loss = criterion(pred.squeeze(-1), y_i)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            model.detach_states()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / n_samples
        run_losses.append(avg_loss)

        print(f"Epoch {epoch + 1}/100 -- train loss: {avg_loss:.6f}")

        # --- Save results after this epoch ---
        log_epoch(run + 1, epoch + 1, avg_loss)
        save_progress(run + 1, epoch + 1, run_losses)

        # Early stopping on train loss (no held-out validation set available)
        if avg_loss < best_train_loss - MIN_DELTA:
            best_train_loss = avg_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"Early stopping at epoch {epoch + 1} (best train loss: {best_train_loss:.6f})")
                break

    model.load_state_dict(best_state)  # restore best-train-loss checkpoint
    model.reset_states(1, device)      # clear stale state before storing

    all_losses.append(run_losses)
    all_models.append(model)
    print(f'Training complete for run {run + 1} (best train loss: {best_train_loss:.6f})')

    predictions, rmse, mse, nmse = evaluate_on_test(
        model, X_test, train, train_scaled, raw_values, scaler, device
    )

    all_rmse.append(rmse)
    all_mse.append(mse)
    all_nmse.append(nmse)
    all_predictions.append(predictions)

    # Save again at run boundary so all_rmse/all_mse/etc. reflect the completed run
    save_progress(run + 1, epoch + 1, run_losses)

    print(f'Run {run + 1} -- RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')

In [ ]:
# Fix the flaky save_progress (retry on Windows file lock)
def save_progress(run, epoch, run_losses_partial):
    snapshot = {
        'all_rmse': all_rmse,
        'all_mse': all_mse,
        'all_nmse': all_nmse,
        'all_predictions': all_predictions,
        'all_losses': all_losses,
        'current_run': run,
        'current_epoch': epoch,
        'current_run_losses_partial': run_losses_partial,
    }
    tmp_path = PROGRESS_PATH + '.tmp'
    with open(tmp_path, 'wb') as f:
        pickle.dump(snapshot, f)
    max_attempts = 5
    for attempt in range(max_attempts):
        try:
            os.replace(tmp_path, PROGRESS_PATH)
            return
        except PermissionError:
            if attempt == max_attempts - 1:
                print(f"  [warning] Could not update progress.pkl this epoch -- continuing.")
                return
            time.sleep(0.2 * (attempt + 1))

# Sanity check before continuing
print(f"Completed runs so far: {len(all_rmse)}")  # should print 40

In [ ]:
resume_from = len(all_rmse)  # 40 — the next run to do
print(f"Resuming from run {resume_from + 1}/{N_RUNS}")

for run in range(resume_from, N_RUNS):
    print(f'\n===== RUN {run + 1}/{N_RUNS} =====')
    np.random.seed(run)
    torch.manual_seed(run)
    model = build_model(input_dim=1, units=8).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    run_losses = []
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    n_samples = X_train_t.size(0)
    best_val_rmse = float('inf')
    best_state = None
    patience_ctr = 0

    for epoch in range(100):
        model.train()
        model.reset_states(1, device)
        epoch_loss = 0.0
        for i in range(n_samples):
            x_i = X_train_t[i:i + 1]
            y_i = y_train_t[i:i + 1]
            optimizer.zero_grad()
            pred = model(x_i)
            loss = criterion(pred.squeeze(-1), y_i)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            model.detach_states()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / n_samples
        run_losses.append(avg_loss)
        _, val_rmse, _, _ = evaluate_on_test(model, val_X, train, train_scaled, raw_values, scaler, device)
        print(f"Epoch {epoch + 1}/100 -- train loss: {avg_loss:.6f}  val RMSE: {val_rmse:.6f}")
        log_epoch(run + 1, epoch + 1, avg_loss, val_rmse)
        save_progress(run + 1, epoch + 1, run_losses)
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"Early stopping at epoch {epoch + 1} (best val RMSE: {best_val_rmse:.6f})")
                break

    model.load_state_dict(best_state)
    model.reset_states(1, device)
    all_losses.append(run_losses)
    all_models.append(model)
    print(f'Training complete for run {run + 1} (best val RMSE: {best_val_rmse:.6f})')
    predictions, rmse, mse, nmse = evaluate_on_test(model, X_test, train, train_scaled, raw_values, scaler, device)
    all_rmse.append(rmse)
    all_mse.append(mse)
    all_nmse.append(nmse)
    all_predictions.append(predictions)
    save_progress(run + 1, epoch + 1, run_losses)
    print(f'Run {run + 1} -- RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')

In [ ]:
#Clean-data summary
print('===== CLEAN-DATA RESULTS (60 runs) =====')
print(f'RMSE: {np.mean(all_rmse):.6f} ± {np.std(all_rmse):.6f}')
print(f'MSE:  {np.mean(all_mse):.6f} ± {np.std(all_mse):.6f}')
print(f'NMSE: {np.mean(all_nmse):.10f} ± {np.std(all_nmse):.10f}')

best_idx = np.argmin(all_rmse)
print(f'\nBest run: {best_idx + 1}')
print(f'  RMSE: {all_rmse[best_idx]:.6f}')
print(f'  MSE:  {all_mse[best_idx]:.6f}')
print(f'  NMSE: {all_nmse[best_idx]:.10f}')

In [ ]:
#Best-run prediction plot
actual = raw_values[-len(test_scaled):]
best_predictions = all_predictions[best_idx]

plt.figure(figsize=(12, 5))
plt.plot(actual, label='Actual', color='blue', linewidth=1.5)
plt.plot(best_predictions, label='Predicted (Best Run)', color='red', linewidth=1.5, linestyle='--')
plt.title('SNN Transformer -- Lake Erie Levels\nPredictions vs Actual (Best of 60 runs, clean data)')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(all_losses[best_idx], color='green', linewidth=1.0)
plt.title('SNN Transformer -- Lake Erie Levels\nTraining Loss (Best Run)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Gaussian Noise-Robustness Sweep (0.5% / 5% / 10% / 15%)

In [ ]:
#Noise-robustness sweep (averaged over multiple noise draws)

N_NOISE_DRAWS = 10  # average over multiple ε(t) realizations per model, per level

noise_robustness = {}  # noise_level -> dict of per-model RMSE/MSE/NMSE (averaged over draws)

for noise_level in NOISE_LEVELS:
    level_rmse, level_mse, level_nmse = [], [], []

    for run, model in enumerate(all_models):
        draw_rmse, draw_mse, draw_nmse = [], [], []
        for draw in range(N_NOISE_DRAWS):
            seed = hash((noise_level, run, draw)) % (2 ** 32)
            X_test_noisy = add_gaussian_noise(X_test, noise_level, ref_std, seed=seed)
            _, rmse, mse, nmse = evaluate_on_test(
                model, X_test_noisy, train, train_scaled, raw_values, scaler, device
            )
            draw_rmse.append(rmse)
            draw_mse.append(mse)
            draw_nmse.append(nmse)

        level_rmse.append(np.mean(draw_rmse))  # this model's average over noise draws
        level_mse.append(np.mean(draw_mse))
        level_nmse.append(np.mean(draw_nmse))

    noise_robustness[noise_level] = {'rmse': level_rmse, 'mse': level_mse, 'nmse': level_nmse}
    print(f"[{noise_level * 100:5.1f}% noise]  "
          f"RMSE: {np.mean(level_rmse):.6f} ± {np.std(level_rmse):.6f}  |  "
          f"MSE: {np.mean(level_mse):.6f} ± {np.std(level_mse):.6f}  |  "
          f"NMSE: {np.mean(level_nmse):.10f} ± {np.std(level_nmse):.10f}")

In [ ]:
from scipy import stats

In [ ]:
# Summary table + paired degradation test
print('===== ROBUSTNESS SUMMARY (60 runs per condition, averaged over 10 noise draws) =====')
print(f"{'Condition':<12}{'RMSE':>14}{'MSE':>16}{'NMSE':>16}")
print(f"{'Clean':<12}{np.mean(all_rmse):>14.6f}{np.mean(all_mse):>16.6f}{np.mean(all_nmse):>16.10f}")
for noise_level in NOISE_LEVELS:
    r = noise_robustness[noise_level]
    label = f"{noise_level * 100:.1f}% noise"
    print(f"{label:<12}{np.mean(r['rmse']):>14.6f}{np.mean(r['mse']):>16.6f}{np.mean(r['nmse']):>16.10f}")

print('\n===== PAIRED DEGRADATION (per-model delta vs its own clean RMSE) =====')
for noise_level in NOISE_LEVELS:
    noisy = np.array(noise_robustness[noise_level]['rmse'])
    clean = np.array(all_rmse)
    assert len(noisy) == len(clean), (
        f"Unpaired arrays at {noise_level}: {len(noisy)} vs {len(clean)}"
    )
    delta = noisy - clean
    t_stat, p_val = stats.ttest_rel(noisy, clean)
    cohens_d = delta.mean() / delta.std(ddof=1) if delta.std(ddof=1) > 0 else float('nan')
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f"{noise_level*100:5.1f}% noise -- ΔRMSE: {delta.mean():.6f} ± {delta.std(ddof=1):.6f}  "
          f"(paired t-test t={t_stat:.3f}, p={p_val:.4f} {sig}, Cohen's d={cohens_d:.3f})")

In [ ]:
#RMSE vs. noise-level plot

labels = ['Clean'] + [f"{n * 100:.1f}%" for n in NOISE_LEVELS]
rmse_means = [np.mean(all_rmse)] + [np.mean(noise_robustness[n]['rmse']) for n in NOISE_LEVELS]
rmse_stds = [np.std(all_rmse)] + [np.std(noise_robustness[n]['rmse']) for n in NOISE_LEVELS]

plt.figure(figsize=(8, 5))
plt.errorbar(labels, rmse_means, yerr=rmse_stds, marker='o', capsize=4, color='darkorange')
plt.title('SNN Transformer -- Lake Erie Levels\nRMSE vs. Input Gaussian Noise Level')
plt.xlabel('Noise level')
plt.ylabel('RMSE (60-run mean ± std)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#Paired RMSE-degradation plot

labels = [f"{n * 100:.1f}%" for n in NOISE_LEVELS]
delta_means = [np.mean(np.array(noise_robustness[n]['rmse']) - np.array(all_rmse)) for n in NOISE_LEVELS]
delta_stds = [np.std(np.array(noise_robustness[n]['rmse']) - np.array(all_rmse)) for n in NOISE_LEVELS]

plt.figure(figsize=(8, 5))
plt.errorbar(labels, delta_means, yerr=delta_stds, marker='o', capsize=4, color='crimson')
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title('SNN Transformer -- Lake Erie Levels\nRMSE Degradation vs. Input Gaussian Noise Level (paired, per-model)')
plt.xlabel('Noise level')
plt.ylabel('ΔRMSE vs. clean baseline (60-run mean ± std)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Summary Statistics (60 runs)
# ============================================================
print('\n===== FINAL RESULTS (60 runs) =====')
print(f'RMSE: {np.mean(all_rmse):.6f} ± {np.std(all_rmse):.6f}')
print(f'MSE:  {np.mean(all_mse):.6f} ± {np.std(all_mse):.6f}')
print(f'NMSE: {np.mean(all_nmse):.10f} ± {np.std(all_nmse):.10f}')
best_idx = np.argmin(all_rmse)
print(f'\nBest run: {best_idx+1}')
print(f'  RMSE: {all_rmse[best_idx]:.6f}')
print(f'  MSE:  {all_mse[best_idx]:.6f}')
print(f'  NMSE: {all_nmse[best_idx]:.10f}')

In [ ]:
# ============================================================
# Predictions vs Actual (Best Run)
# ============================================================
best_predictions = all_predictions[best_idx]
actual = raw_values[-len(best_predictions):]

plt.figure(figsize=(12, 5))
plt.plot(actual, label='Actual', color='blue', linewidth=1.5)
plt.plot(best_predictions, label='Predicted (Best Run)', color='red',
         linewidth=1.5, linestyle='--')
plt.title('Fuzzy Output Layer — sp500\nPredictions vs Actual (Best of 60 runs)')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# Loss Curve (Best Run)
# ============================================================
plt.figure(figsize=(12, 4))
plt.plot(all_losses[best_idx], color='green', linewidth=1.0)
plt.title('Fuzzy Output Layer — sp500\nTraining Loss (Best Run)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# Final Metrics Summary
# ============================================================
print('=== Best Run Metrics ===')
print(f'RMSE: {all_rmse[best_idx]:.6f}')
print(f'MSE:  {all_mse[best_idx]:.6f}')
print(f'NMSE: {all_nmse[best_idx]:.10f}')

In [ ]:
# ============================================================
# NOTEBOOK TIMER — END
# ============================================================
import time
_NOTEBOOK_END_TIME = time.time()
try:
    _NOTEBOOK_ELAPSED = _NOTEBOOK_END_TIME - _NOTEBOOK_START_TIME
    print(f"\nNotebook execution complete.\nTotal Elapsed Time: {_NOTEBOOK_ELAPSED/60:.2f} minutes")
except NameError:
    pass
